# PySofra — End-to-End Tutorial

> *The missing statistical reporting layer for Python.*

PySofra turns datasets, fitted models, and summary statistics into
**publication-ready tables** — across HTML, Markdown, DOCX, LaTeX,
PPTX, and Excel — from a single immutable `SofraTable` object. This
tutorial walks through every public feature with realistic examples
on a synthetic two-arm clinical trial.

**Where PySofra fits in the pipeline.** PySofra picks up where your
analysis is done — cleaning, transformations, outlier handling,
imputation, and model fitting stay with `pandas` / `scikit-learn` /
`statsmodels` / `lifelines`. PySofra's job is to take those finished
results to a publication-ready table. Every methodological choice
(which variables are non-normal, which test to use, what the survey
design looks like) is an **explicit keyword in the call**, not a
hidden auto-decision — so your methods section reads unambiguously
from the code, and the same input always produces the same table.

### What this tutorial covers

| # | Section |
|---:|---|
| 1 | Setup |
| 2 | Table 1 — baseline characteristics |
| 3 | The statistics PySofra is actually running |
| 4 | New in this release — four capabilities |
| 5 | Custom labels, type overrides, and column ordering |
| 6 | Non-normal continuous variables |
| 7 | Handling missing data |
| 8 | `tbl_summary` — descriptive summaries without stratification |
| 9 | Regression results — logistic |
| 10 | Regression results — OLS (linear) |
| 11 | Highlighting significant effects — `bold_p` |
| 12 | Composing tables side-by-side — `tbl_merge` |
| 13 | Stacking tables vertically — `tbl_stack` |
| 14 | Per-variable test overrides |
| 15 | Multiplicity adjustment — `.add_q()` |
| 16 | Side-by-side multi-model regression |
| 17 | Survival regression — Cox (lifelines) |
| 18 | Polars input |
| 19 | Kaplan–Meier summary — `tbl_survival` |
| 20 | Survey-weighted Table 1 |
| 21 | Complex survey designs — `SurveyDesign` |
| 22 | Univariable regression — `tbl_uvregression` |
| 23 | Differences, CIs, joint p-values |
| 24 | Custom formatters & in-prose extraction |
| 25 | Visual previews — table-as-image + KM plot |
| 26 | Rich-cell composition — `compose()` + `CellPart` |
| 27 | Calibration — `post_stratify` / `rake` |
| 28 | Cross-tabulation — `tbl_cross` |
| 29 | Effect sizes & one-line decorators |
| 30 | Multiple-imputation pooling (Rubin's rules) |
| 31 | Themes |
| 32 | Captions, footnotes, and custom themes |
| 33 | Conditional row formatting |
| 34 | Sticky headers for long tables |
| 35 | Inline plots — forest + Kaplan–Meier |
| 36 | Exports |
| 37 | Under the hood — the `SofraTable` object |


> Every output renders live in the notebook so you can see exactly what
> will appear in your manuscript before you export.

---


## 1. Setup

PySofra has only one required runtime: a working Python ≥ 3.11 with
`numpy`, `pandas`, `scipy`, `statsmodels`, and `python-docx`. Optional
extras (`polars`, `lifelines`, `python-pptx`, `xlsxwriter`) unlock the
features that depend on them. The block below imports the standard
public API.


In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

import pysofra as ps

print('PySofra', ps.__version__)
print('Built-in themes:', ps.available_themes())

PySofra 0.1.0a16
Built-in themes: ['clinical', 'compact', 'default', 'jama', 'minimal', 'nejm']


### A synthetic clinical trial

The rest of the tutorial uses a 300-patient two-arm dataset:
**Placebo** vs **Treatment**, six baseline characteristics, and one
binary outcome. We seed `numpy`'s default RNG so every output is
reproducible to the digit.


In [2]:
rng = np.random.default_rng(2026_05_20)
n = 300

df = pd.DataFrame({
    'arm':    rng.choice(['Placebo', 'Treatment'], n, p=[0.5, 0.5]),
    'age':    rng.normal(62, 11, n).round(1),
    'bmi':    rng.normal(28,  5, n).round(1),
    'sex':    rng.choice(['Female', 'Male'], n, p=[0.55, 0.45]),
    'smoker': rng.choice([0, 1], n, p=[0.7, 0.3]),
    'race':   rng.choice(['White', 'Black', 'Asian', 'Other'], n,
                         p=[0.6, 0.2, 0.15, 0.05]),
    'hba1c':  rng.lognormal(mean=np.log(6.5), sigma=0.18, size=n).round(2),
})
# Outcome with a real age effect to make regression interesting.
linpred = -6 + 0.07 * df['age'] + 0.04 * df['bmi'] - 0.3 * (df['sex'] == 'Male')
df['event'] = (rng.uniform(0, 1, n) < 1.0 / (1.0 + np.exp(-linpred))).astype(int)

# Sprinkle in a little missingness on BMI.
df.loc[rng.choice(df.index, 8, replace=False), 'bmi'] = np.nan

df.head()

,arm,age,bmi,sex,smoker,race,hba1c,event
0,Placebo,47.7,27.0,Female,0,White,7.95,0
1,Placebo,59.2,29.3,Male,1,White,7.51,0
2,Placebo,49.4,36.8,Female,0,Other,7.15,0
3,Placebo,59.9,40.7,Female,0,White,7.04,1
4,Treatment,68.5,35.3,Male,0,White,6.79,0


## 2. Table 1 — baseline characteristics

> The bread-and-butter call. Build a publication-grade Table 1 in one
> line.

`tbl_one(df, by='arm')` produces a fully formatted baseline table:
per-arm summaries, an automatically selected statistical test per
variable, and standardised mean differences. Every modifier method
(`.add_p()`, `.add_smd()`, `.add_overall()`, `.theme()`) returns a
**new** immutable `SofraTable` — call them in any order, chain freely.


In [3]:
(
    ps.tbl_one(df, by='arm')
      .add_p()
      .add_smd()
      .add_overall()
      .theme('clinical')
)

Characteristic,OverallN = 300,PlaceboN = 145,TreatmentN = 155,p-value,SMD
age,61.79 (10.42),61.18 (10.68),62.36 (10.17),0.330,0.113
bmi,28.08 (4.97),28.38 (4.94),27.80 (5.00),0.321,0.116
Missing,8 (2.7%),2 (1.4%),6 (3.9%),,
sex = Male,131 (43.7%),61 (42.1%),70 (45.2%),0.642,0.062
smoker = 1,97 (32.3%),52 (35.9%),45 (29.0%),0.219,0.146
race,,,,0.749,0.128
Asian,55 (18.3%),27 (18.6%),28 (18.1%),,
Black,61 (20.3%),32 (22.1%),29 (18.7%),,
Other,16 (5.3%),6 (4.1%),10 (6.5%),,
White,168 (56.0%),80 (55.2%),88 (56.8%),,


### Reading the output

* `age`, `bmi`, `hba1c` were auto-typed **continuous** → `Mean (SD)`.
* `sex` has two levels → **dichotomous**, rendered as `sex = Male`.
* `smoker` is `0/1` → also dichotomous.
* `race` has four levels → **categorical**, one indented row per level.
* `event` is `0/1` → dichotomous.
* The test for each row was chosen automatically (Welch's *t* for
  continuous + Fisher's exact / χ² for categorical). The footnote lists
  them.

---


## 3. The statistics PySofra is actually running

> **PySofra is not a formatter that someone else's numbers pass through.**
> It picks the right test for each variable, runs it via `scipy` /
> `statsmodels` / `lifelines`, and surfaces the result in the cell. This
> section opens the hood: every test you've already seen rendered above
> is reproduced *byte-for-byte* by a direct call to the underlying
> library on the same data.

### Auto-dispatch rules

| Variable kind | Group count | Default test | `nonnormal=[...]` switch |
|---|---|---|---|
| Continuous | 2 | **Welch's *t*-test** | Wilcoxon rank-sum (Mann–Whitney *U*) |
| Continuous | 3+ | **One-way ANOVA** | Kruskal–Wallis |
| Categorical 2×2 | 2 | **Fisher's exact** | — |
| Categorical R×C | 2+ | **Pearson χ²** (flagged sparse if any expected < 5) | — |
| Continuous, weights= | 2 | **Design-adjusted *t*-test** (Taylor linearisation) | — |
| Categorical, weights= | 2+ | **Rao–Scott first-order χ²** | — |

Per-variable overrides are passed via `tests={'age': 'wilcoxon', ...}`.
The exact dispatch table is queryable at runtime:


In [4]:
ps.available_tests()

{'continuous': ['anova',
  'equal_var_t',
  'kruskal',
  'kruskal_wallis',
  'mannwhitney',
  'mwu',
  'oneway_anova',
  'rank_sum',
  'student',
  'student_t',
  't',
  'ttest',
  'welch',
  'welch_t',
  'wilcoxon'],
 'categorical': ['chi2',
  'chi_square',
  'chisq',
  'fisher',
  'fisher_exact',
  'pearson']}

### 3.1 Welch's *t*-test on `age` — pysofra vs. raw scipy

The Table 1 above showed a *p*-value for `age` from Welch's two-sample
*t*-test. Below: the pysofra cell and the direct `scipy.stats.ttest_ind`
call on exactly the same arrays. The numbers are identical to machine
precision.


In [5]:
from scipy import stats as sp_stats

# 1) The pysofra-reported p-value.
tbl_p = ps.tbl_one(df, by='arm').add_p()
age_row = next(r for r in tbl_p.rows if r.cells[0].text == 'age')
p_pysofra = next(c.value for c in age_row.cells if c.kind == 'p_value')

# 2) The same Welch t-test computed directly with scipy.
a = df.loc[df['arm'] == 'Placebo',   'age'].dropna().to_numpy()
b = df.loc[df['arm'] == 'Treatment', 'age'].dropna().to_numpy()
_, p_scipy = sp_stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')

print(f'pysofra p (age, Welch) = {p_pysofra!r}')
print(f'scipy   p (age, Welch) = {p_scipy!r}')
print(f'identical Python floats? {p_pysofra == float(p_scipy)}')


pysofra p (age, Welch) = 0.329576311105047
scipy   p (age, Welch) = np.float64(0.329576311105047)
identical Python floats? True


### 3.2 Fisher's exact on `sex` — pysofra vs. raw scipy

The 2×2 categorical case dispatches to Fisher's exact (not chi-square).
Same proof.


In [6]:
sex_row = next(r for r in tbl_p.rows if r.cells[0].text == 'sex = Male')
p_pysofra = next(c.value for c in sex_row.cells if c.kind == 'p_value')

ctab = pd.crosstab(df['sex'], df['arm']).to_numpy()
_, p_scipy = sp_stats.fisher_exact(ctab, alternative='two-sided')

print(f'pysofra p (sex, Fisher) = {p_pysofra:.10g}')
print(f'scipy   p (sex, Fisher) = {p_scipy:.10g}')


pysofra p (sex, Fisher) = 0.6416844023
scipy   p (sex, Fisher) = 0.6416844023


### 3.3 `nonnormal=` switches the dispatch to Wilcoxon

For variables flagged non-normal, the dispatch swaps Welch out for the
Wilcoxon rank-sum (Mann–Whitney *U*) test. The summary statistic in the
cell also switches from `Mean (SD)` to `Median (Q1, Q3)`.


In [7]:
tbl_nn = ps.tbl_one(df, by='arm', nonnormal=['hba1c']).add_p()
hba1c_row = next(r for r in tbl_nn.rows if r.cells[0].text == 'hba1c')
p_pysofra = next(c.value for c in hba1c_row.cells if c.kind == 'p_value')

a = df.loc[df['arm'] == 'Placebo',   'hba1c'].dropna().to_numpy()
b = df.loc[df['arm'] == 'Treatment', 'hba1c'].dropna().to_numpy()
_, p_scipy = sp_stats.mannwhitneyu(a, b, alternative='two-sided')

print(f'pysofra p (hba1c, Wilcoxon) = {p_pysofra:.10g}')
print(f'scipy   p (hba1c, Wilcoxon) = {p_scipy:.10g}')


pysofra p (hba1c, Wilcoxon) = 0.6160530199
scipy   p (hba1c, Wilcoxon) = 0.6160530199


### 3.4 Per-variable test override via `tests={...}`

The auto-dispatch is the default, not a constraint. Any variable can be
forced onto any test by name. Same `age` column, two different tests,
two different *p*-values — both correct, both reproducible.


In [8]:
t_default  = ps.tbl_one(df, by='arm', variables=['age']).add_p()
t_override = ps.tbl_one(df, by='arm', variables=['age'],
                        tests={'age': 'wilcoxon'}).add_p()

def get_p(tbl):
    row = next(r for r in tbl.rows if r.cells[0].text == 'age')
    return next(c.value for c in row.cells if c.kind == 'p_value')

print(f"default  (Welch's t)        : p = {get_p(t_default):.6g}")
print(f"override (Wilcoxon rank-sum): p = {get_p(t_override):.6g}")
print()
print('available test keys:', list(ps.available_tests()['continuous']))


default  (Welch's t)        : p = 0.329576
override (Wilcoxon rank-sum): p = 0.160578

available test keys: ['anova', 'equal_var_t', 'kruskal', 'kruskal_wallis', 'mannwhitney', 'mwu', 'oneway_anova', 'rank_sum', 'student', 'student_t', 't', 'ttest', 'welch', 'welch_t', 'wilcoxon']


### 3.5 Standardized mean differences — textbook formula, exactly

`.add_smd()` adds a standardized mean difference column. For a continuous
variable that's

$$\mathrm{SMD} = \frac{\bar{x}_{\mathrm{A}} - \bar{x}_{\mathrm{B}}}
                       {\sqrt{(s_{\mathrm{A}}^2 + s_{\mathrm{B}}^2) / 2}}$$

For a categorical variable with $K$ levels, PySofra uses the
**Yang–Dalton** multivariate formulation (the standard in propensity-score
matching literature). Both verified below.


In [9]:
from pysofra.summary.smd import continuous_smd_pair, categorical_smd_pair

tbl_smd = ps.tbl_one(df, by='arm', variables=['age', 'race']).add_smd()
smd_col = next(i for i, h in enumerate(tbl_smd.headers[0].cells) if h.text == 'SMD')

# 1) Continuous SMD verified against the textbook formula AND against the
#    same primitive PySofra uses internally (continuous_smd_pair).
a = df.loc[df['arm'] == 'Placebo',   'age'].dropna().to_numpy()
b = df.loc[df['arm'] == 'Treatment', 'age'].dropna().to_numpy()
# PySofra's convention orders groups alphabetically and reports the
# absolute SMD (the clinical-literature standard).
smd_textbook = abs(a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
smd_primitive = continuous_smd_pair(a, b)

age_row = next(r for r in tbl_smd.rows if r.cells[0].text == 'age')
smd_pysofra = age_row.cells[smd_col].value
print('continuous SMD (age):')
print(f'    table cell        = {smd_pysofra:+.6f}')
print(f'    textbook formula  = {smd_textbook:+.6f}')
print(f'    continuous_smd_pair = {smd_primitive:+.6f}')

# 2) Categorical SMD via the public Yang-Dalton primitive. The primitive
#    takes *proportion vectors* over a common set of K levels, so build
#    those first from the raw race data.
levels = sorted(df['race'].dropna().unique())
p1 = np.array([(df.loc[df['arm'] == 'Placebo',   'race'] == lvl).mean() for lvl in levels])
p2 = np.array([(df.loc[df['arm'] == 'Treatment', 'race'] == lvl).mean() for lvl in levels])
smd_yd = categorical_smd_pair(p1, p2)

race_row = next(r for r in tbl_smd.rows if r.cells[0].text == 'race')
smd_pysofra = race_row.cells[smd_col].value
print()
print('categorical SMD (race):')
print(f'    table cell        = {smd_pysofra:+.6f}')
print(f'    Yang-Dalton (categorical_smd_pair) = {smd_yd:+.6f}')


continuous SMD (age):
    table cell        = +0.112923
    textbook formula  = +0.112923
    continuous_smd_pair = +0.112923

categorical SMD (race):
    table cell        = +0.127822
    Yang-Dalton (categorical_smd_pair) = +0.127822


### 3.6 Multiplicity adjustment — pysofra vs. `statsmodels.stats.multitest`

`.add_q()` adjusts the family of *p*-values for multiple testing.
Internally it calls `statsmodels.stats.multitest.multipletests` — the
*q*-values you see in the table are exactly what `multipletests`
returns, in the same order.


In [10]:
from statsmodels.stats.multitest import multipletests

tbl_q = ps.tbl_one(df, by='arm').add_p().add_q(method='fdr_bh')

p_pysofra = [c.value for r in tbl_q.rows
                       for c in r.cells if c.kind == 'p_value'
                       and isinstance(c.value, (int, float))]
q_pysofra = [c.value for r in tbl_q.rows
                       for c in r.cells if c.kind == 'q_value'
                       and isinstance(c.value, (int, float))]

_, q_sm, _, _ = multipletests(p_pysofra, method='fdr_bh')

print(f'{"variable p":>16}   {"pysofra q":>10}   {"statsmodels q":>14}')
for p, qa, qb in zip(p_pysofra, q_pysofra, q_sm):
    print(f'{p:16.6f}   {qa:10.6f}   {qb:14.6f}')


      variable p    pysofra q    statsmodels q
        0.329576     0.749483         0.749483
        0.321470     0.749483         0.749483
        0.641684     0.749483         0.749483
        0.218755     0.749483         0.749483
        0.749483     0.749483         0.749483
        0.734512     0.749483         0.749483
        0.718454     0.749483         0.749483


### 3.7 Multiple-imputation pooling — Rubin's rules from scratch

`pool([fit_1, fit_2, ...])` combines $m$ analyses fit on $m$ imputed
datasets via Rubin's rules:

$$\bar{\beta} = \tfrac{1}{m}\sum \hat\beta_k, \quad
  T = \bar U + (1 + 1/m)\, B$$

where $\bar U$ is the average within-imputation variance and $B$ is
the between-imputation variance. The pooled point estimate is just the
mean of the per-imputation estimates; the pooled SE follows from $T$.
Verified by hand below.


In [11]:
# Build 3 toy imputations of bmi (which has 8 missing) by sampling
# from the conditional mean given age. Fit logistic event ~ age + bmi
# on each, then pool.
import statsmodels.api as sm

def impute_once(seed):
    d = df.copy()
    miss = d['bmi'].isna()
    rng_local = np.random.default_rng(seed)
    noise = rng_local.normal(0, d['bmi'].std(), miss.sum())
    d.loc[miss, 'bmi'] = d['bmi'].mean() + noise
    X = sm.add_constant(d[['age', 'bmi']])
    return sm.Logit(d['event'], X).fit(disp=False)

fits = [impute_once(s) for s in (1, 2, 3)]
pooled = ps.pool(fits)            # returns a ModelSummary
tbl_pooled = ps.tbl_regression(pooled)

# Verify pooled estimate = mean of per-imputation estimates (Rubin Rule 1).
betas_age = np.array([f.params['age'] for f in fits])
ses_age   = np.array([f.bse['age']    for f in fits])
m = len(fits)
beta_bar = betas_age.mean()
U_bar    = (ses_age ** 2).mean()
B        = betas_age.var(ddof=1)
T_total  = U_bar + (1 + 1.0 / m) * B
se_pooled = np.sqrt(T_total)

beta_pysofra = float(pooled.estimates['age'])
p_pysofra    = float(pooled.pvalues['age'])

print(f'pooled β(age):  pysofra={beta_pysofra:+.6f}   Rubin manual={beta_bar:+.6f}')
print(f'pooled SE:                                 Rubin manual={se_pooled:+.6f}')
print(f'  (within-imputation variance Ū = {U_bar:.6g}, '
      f'between-imputation variance B = {B:.6g}, '
      f'total variance T = {T_total:.6g})')
print()
print('rendered as a SofraTable via ps.tbl_regression(pooled):')
tbl_pooled


pooled β(age):  pysofra=+0.065188   Rubin manual=+0.065188
pooled SE:                                 Rubin manual=+0.013181
  (within-imputation variance Ū = 0.00017359, between-imputation variance B = 1.14515e-07, total variance T = 0.000173743)

rendered as a SofraTable via ps.tbl_regression(pooled):


SofraTable(rows=2, cols=4, theme='default')

### 3.8 Survey-weighted dispatch — Rao–Scott $\chi^2$ and design-adjusted *t*

When `weights=` is passed to `tbl_one`, the **categorical** dispatch
swaps Pearson $\chi^2$ for the **Rao–Scott first-order corrected
$\chi^2$** (Pearson statistic on the *weighted* contingency table,
divided by an estimated design effect). Compare the two footnotes
below — only the categorical test label changes.

For a **design-adjusted *t*-test** on a continuous variable (Taylor
linearisation of the weighted mean difference), pass a full
`SurveyDesign(weights=, strata=, cluster=)` — the design object carries
enough information to compute the linearised variance. The
auto-dispatch table for $t$-tests requires a `SurveyDesign`, not just
a weight column, to switch.


In [12]:
rng2 = np.random.default_rng(11)
df_w = df.copy()
df_w['w'] = rng2.uniform(0.5, 2.0, len(df_w))   # inverse-probability weights

# Unweighted: Pearson chi-square for categorical, Welch's t for continuous.
tbl_uw = ps.tbl_one(df_w, by='arm', variables=['age', 'race']).add_p()
print('UNWEIGHTED  ', tbl_uw.footnotes[-1])

# Weighted: Rao-Scott chi-square auto-selected for the categorical row.
tbl_sw = ps.tbl_one(df_w, by='arm', weights='w',
                    variables=['age', 'race']).add_p()
print('WEIGHTED    ', tbl_sw.footnotes[-1])


UNWEIGHTED   Tests: Pearson's chi-square; Welch's t-test.
WEIGHTED     Tests: Design-adjusted t-test; Rao–Scott chi-square.


### 3.9 Regression *p*-values flow straight from the model fit

For `tbl_regression`, PySofra does **not** recompute *p*-values — it
reads them from the fitted model and renders them. Same for coefficients
and standard errors. The proof:


In [13]:
# Drop the rows with NaN bmi for a clean fit.
df_fit = df.dropna(subset=['bmi']).copy()
X = sm.add_constant(df_fit[['age', 'bmi']])
fit = sm.Logit(df_fit['event'], X).fit(disp=False)

tbl_reg = ps.tbl_regression(fit, exponentiate=True)

print(f'{"predictor":>10}   {"model p":>12}   {"table p":>12}')
for name in ('age', 'bmi'):
    row = next(r for r in tbl_reg.rows if r.cells[0].text == name)
    p_table = next(c.value for c in row.cells if c.kind == 'p_value')
    print(f'{name:>10}   {fit.pvalues[name]:12.8f}   {p_table:12.8f}')


 predictor        model p        table p
       age     0.00000034     0.00000034
       bmi     0.18265302     0.18265302


### Summary

Every *p*-value, every SMD, every *q*-value, and every effect size in
this tutorial is reproducible to machine precision by a direct call to
`scipy`, `statsmodels`, or `lifelines` on the same data. PySofra's job
is to **pick the right test for the variable** and **render the result
in publication form** — the numerics belong to the underlying library.

The full numeric-agreement cross-check across **21 primitives against
R's `gtsummary` package** is in
[the test suite](../the test suite)
— it reports ``.

---


## 4. Statistical modifier highlights

> These cells verify each statistical modifier against direct `scipy` /
> `statsmodels` calls on the same data — PySofra dispatches, the
> numerics belong to the underlying library.

| § | Capability | Path |
|---:|---|---|
| 4.1 | `weights=` alone now picks the **design-adjusted *t*-test** | continuous variable + survey weights |
| 4.2 | `tbl_cross` accepts **`.add_p()` and `.add_overall()`** | contingency table with auto-selected χ² / Fisher's exact |
| 4.3 | `.add_global_p()` works on **`tbl_one` / `tbl_summary`** (optionally `adjust_for=`) | joint Wald *p* per variable, covariate-adjusted on request |
| 4.4 | `tbl_regression` accepts **sklearn multi-class** classifiers | one row per (class, feature) pair |


### 4.1 `weights=` auto-promotes to a design-adjusted *t*-test

Previously, passing `weights=` to `tbl_one` only switched the
*categorical* test to Rao–Scott χ². The *continuous* test silently
fell back to **unweighted Welch** — which discards the weights. As of
this release, `weights=` alone triggers the **design-adjusted
*t*-test** (Taylor-linearised; `svyttest` analogue) for the 2-arm
continuous case.

The footnote names the actual test used, and the *p*-value matches a
direct call to the `svyttest` primitive to machine precision.


In [14]:
# Add inverse-probability weights to the existing fixture.
rng2 = np.random.default_rng(11)
df_w = df.copy()
df_w['w'] = rng2.uniform(0.5, 2.0, len(df_w))

# Unweighted: Welch's t for continuous.
t_unweighted = ps.tbl_one(df_w, by='arm', variables=['age']).add_p()
print('UNWEIGHTED footnote:', t_unweighted.footnotes[-1])

# Weighted: auto-promotes to the design-adjusted t-test.
t_weighted = ps.tbl_one(df_w, by='arm', weights='w',
                        variables=['age']).add_p()
print('WEIGHTED footnote:  ', t_weighted.footnotes[-1])

# Cross-check: the weighted p-value must equal svyttest() directly.
from pysofra.summary.tests import svyttest
res = svyttest(df_w['age'], df_w['arm'], df_w['w'])
p_table = next(c.value for r in t_weighted.rows
               if r.cells[0].text == 'age'
               for c in r.cells if c.kind == 'p_value')
print()
print(f'pysofra age p (weighted): {p_table!r}')
print(f'svyttest()      directly: {float(res.p_value)!r}')
print(f'identical: {p_table == float(res.p_value)}')


UNWEIGHTED footnote: Tests: Welch's t-test.
WEIGHTED footnote:   Tests: Design-adjusted t-test.

pysofra age p (weighted): 0.36992784142393725
svyttest()      directly: 0.36992784142393725
identical: True


### 4.2 `tbl_cross` accepts `.add_p()` and `.add_overall()`

`tbl_cross` now carries a rebuild closure, so statistical modifiers
work directly on the contingency table:

* `.add_p()` runs the auto-selected categorical test on the full
  contingency (Fisher's exact for 2×2, Pearson χ² otherwise) and adds
  the result as a footnote.
* `.add_overall()` toggles margins on (no-op when margins are already
  the default).
* `.add_smd()` raises `NotImplementedError` — SMD is the standardised
  difference between two distributions of a *single* variable and is
  undefined on a contingency. Pointing that out is the right behaviour
  rather than silently emitting a misleading column.

Below: pysofra's footnote *p* matches the direct `scipy.stats`
calculation.


In [15]:
# 2x2 contingency: sex by arm. tbl_cross auto-picks Fisher's exact.
t = ps.tbl_cross(df, row='sex', column='arm').add_p()
print('footnotes:', t.footnotes)
print('metadata: ', {k: v for k, v in t.metadata.items() if k != 'builder'})

# Cross-check Fisher's exact against scipy directly.
ctab = pd.crosstab(df['sex'], df['arm']).to_numpy()
_, p_scipy = sp_stats.fisher_exact(ctab, alternative='two-sided')
print(f'\npysofra p: {t.metadata["p_value"]!r}')
print(f'scipy   p: {p_scipy!r}')
print(f'identical: {t.metadata["p_value"] == float(p_scipy)}')

# Show the rendered table.
t


footnotes: ('Cells: n (column-%).', "Fisher's exact: p = 0.642")
metadata:  {'p_value': 0.6416844022938752, 'p_test': "Fisher's exact"}

pysofra p: 0.6416844022938752
scipy   p: np.float64(0.6416844022938752)
identical: True


SofraTable(rows=3, cols=4, theme='default')

In [16]:
# RxC table: race by arm. 3+ rows -> auto-switches to Pearson chi-square.
t = ps.tbl_cross(df, row='race', column='arm').add_p()
print(t.footnotes[-1])

# And the .add_smd() informative error:
try:
    ps.tbl_cross(df, row='race', column='arm').add_smd()
except NotImplementedError as e:
    print('\n.add_smd() raised:')
    print(' ', str(e)[:120], '...')


Pearson's chi-square: p = 0.749

.add_smd() raised:
  add_smd() is not defined for tbl_cross — SMD measures the standardised difference between two distributions of a single  ...


### 4.3 `.add_global_p()` on `tbl_one` — joint Wald, optionally covariate-adjusted

Previously this raised `NotImplementedError`. Now it fits one logistic
regression per variable on the source data:

$$\mathrm{Logit}(\mathrm{arm} = \mathrm{ref} \sim \mathrm{variable} \,
[+ \mathrm{adjust\_for}])$$

and reports the **joint Wald *p*-value** across that variable's
coefficients. For a multi-level categorical (e.g. `race` with four
levels = three dummies), this is a 3-degree-of-freedom test — the same
"global *p*" that gtsummary's `add_global_p()` produces.

Pass `adjust_for=['age', 'bmi']` and every per-variable regression
includes those covariates, giving covariate-adjusted joint *p*-values
in one line.


In [17]:
# Unadjusted global p per variable.
t = ps.tbl_one(df, by='arm').add_global_p()
t


Characteristic,PlaceboN = 145,TreatmentN = 155,global p
age,61.18 (10.68),62.36 (10.17),0.329
bmi,28.38 (4.94),27.80 (5.00),0.322
Missing,2 (1.4%),6 (3.9%),
sex = Male,61 (42.1%),70 (45.2%),0.590
smoker = 1,52 (35.9%),45 (29.0%),0.208
race,,,0.753
Asian,27 (18.6%),28 (18.1%),
Black,32 (22.1%),29 (18.7%),
Other,6 (4.1%),10 (6.5%),
White,80 (55.2%),88 (56.8%),


In [18]:
# Covariate-adjusted global p — same statistic but each per-variable
# Logit now includes age + bmi as covariates.
t_adj = (
    ps.tbl_one(df, by='arm', variables=['sex', 'race'])
      .add_global_p(adjust_for=['age', 'bmi'])
)
t_adj


SofraTable(rows=6, cols=4, theme='default')

In [19]:
# Cross-check: pysofra's race global p must match a direct
# statsmodels Logit(arm ~ C(race)) joint Wald F-test.
import statsmodels.api as sm

sub = df[['arm', 'race']].dropna()
y = (sub['arm'] == 'Treatment').astype(int).to_numpy()
dummies = pd.get_dummies(sub['race'], prefix='race', drop_first=True,
                         dtype=float).to_numpy()
X = sm.add_constant(dummies)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    res = sm.Logit(y, X).fit(disp=False)

n_dummies = X.shape[1] - 1
constraint = ', '.join(f'x{i+1} = 0' for i in range(n_dummies))
p_direct = float(res.f_test(constraint).pvalue)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    t = ps.tbl_one(df, by='arm', variables=['race']).add_global_p()
p_table = next(r for r in t.rows if r.cells[0].text == 'race').cells[-1].value

print(f'pysofra race global p:    {p_table!r}')
print(f'statsmodels f_test:       {p_direct!r}')
print(f'identical Python floats?  {p_table == p_direct}')


pysofra race global p:    0.7527548796583019
statsmodels f_test:       0.7527548796583019
identical Python floats?  True


### 4.4 `tbl_regression` accepts sklearn multi-class classifiers

Previously raised `NotImplementedError` on
`model.coef_.shape[0] != 1`. Now `tbl_regression` flattens the
$(n_\text{classes} \times n_\text{features})$ coefficient matrix into one row per
$(\text{class}, \text{feature})$ pair using the same `"feature (class=X)"` flat-label
convention as lifelines AFT models.

The exponentiated OR for each row equals `exp(model.coef_[class_idx,
feature_idx])` exactly.


In [20]:
# Synthetic 3-class outcome (mild / moderate / severe).
from sklearn.linear_model import LogisticRegression

rng_mc = np.random.default_rng(0)
X_mc = pd.DataFrame({
    'age': rng_mc.normal(60, 10, 400),
    'bmi': rng_mc.normal(28, 5, 400),
})
y_mc = pd.Series(rng_mc.choice(['mild', 'moderate', 'severe'], 400,
                                p=[0.5, 0.3, 0.2]))

clf = LogisticRegression(max_iter=1000).fit(X_mc, y_mc)
print(f'coef_ shape: {clf.coef_.shape}  (n_classes={clf.coef_.shape[0]}, '
      f'n_features={clf.coef_.shape[1]})')
print(f'classes_:    {list(clf.classes_)}')

tbl = ps.tbl_regression(clf, exponentiate=True)
tbl


coef_ shape: (3, 2)  (n_classes=3, n_features=2)
classes_:    ['mild', 'moderate', 'severe']


SofraTable(rows=6, cols=4, theme='default')

In [21]:
# Cross-check: every rendered OR must equal exp(model.coef_[i, j])
# on the matching (class, feature) row.
from pysofra.models.extract import _extract_sklearn

ms = _extract_sklearn(clf)
mismatches = 0
for ci, cls in enumerate(clf.classes_):
    for fi, feat in enumerate(['age', 'bmi']):
        key = f'{feat} (class={cls})'
        got  = ms.estimates[key]
        want = clf.coef_[ci, fi]
        if abs(got - want) > 1e-12:
            mismatches += 1
print(f'(class, feature) pairs: {len(clf.classes_) * 2}')
print(f'mismatches vs model.coef_: {mismatches}')


(class, feature) pairs: 6
mismatches vs model.coef_: 0


---

## 5. Custom labels, type overrides, and column ordering

Real Table 1s rarely use raw column names. Pass `labels=` to rename,
`types=` to override the inferred kind, and `variables=` to control
both **inclusion** and **order**.


In [22]:
(
    ps.tbl_one(
        df,
        by='arm',
        variables=['age', 'sex', 'bmi', 'smoker', 'race', 'hba1c'],
        labels={
            'age':    'Age (years)',
            'bmi':    'BMI (kg/m\u00b2)',
            'sex':    'Sex',
            'smoker': 'Current smoker',
            'race':   'Race',
            'hba1c':  'HbA1c (%)',
        },
        # Force smoker to be reported as categorical with named levels.
        types={'smoker': 'categorical'},
    )
    .add_p()
    .add_smd()
    .theme('clinical')
)

Characteristic,PlaceboN = 145,TreatmentN = 155,p-value,SMD
Age (years),61.18 (10.68),62.36 (10.17),0.330,0.113
Sex = Male,61 (42.1%),70 (45.2%),0.642,0.062
BMI (kg/m²),28.38 (4.94),27.80 (5.00),0.321,0.116
Missing,2 (1.4%),6 (3.9%),,
Current smoker,,,0.219,0.146
0,93 (64.1%),110 (71.0%),,
1,52 (35.9%),45 (29.0%),,
Race,,,0.749,0.128
Asian,27 (18.6%),28 (18.1%),,
Black,32 (22.1%),29 (18.7%),,


## 6. Non-normal continuous variables

> Skewed → median (Q1, Q3) → rank-based test.

Pass column names to `nonnormal=` to switch the summary to
**median (Q1, Q3)** and the test to Wilcoxon (2 groups) or
Kruskal–Wallis (3+ groups). HbA1c is mildly right-skewed, so this is
the textbook choice.


In [23]:
(
    ps.tbl_one(df, by='arm', nonnormal=['hba1c'])
      .add_p()
      .add_smd()
      .theme('clinical')
)

Characteristic,PlaceboN = 145,TreatmentN = 155,p-value,SMD
age,61.18 (10.68),62.36 (10.17),0.330,0.113
bmi,28.38 (4.94),27.80 (5.00),0.321,0.116
Missing,2 (1.4%),6 (3.9%),,
sex = Male,61 (42.1%),70 (45.2%),0.642,0.062
smoker = 1,52 (35.9%),45 (29.0%),0.219,0.146
race,,,0.749,0.128
Asian,27 (18.6%),28 (18.1%),,
Black,32 (22.1%),29 (18.7%),,
Other,6 (4.1%),10 (6.5%),,
White,80 (55.2%),88 (56.8%),,


## 7. Handling missing data

`missing=` controls how the summary table reports `NaN`:

* `'ifany'` (default) — show a *Missing* row only for variables that
  actually have missing values.
* `'always'` — show a *Missing* row for every variable, even when zero.
* `'never'` — drop the *Missing* row entirely.

Both alternatives below are useful: `'always'` for protocol-locked
deliverables; `'never'` for a clean cosmetic look.


In [24]:
(
    ps.tbl_one(df, by='arm', missing='always',
               variables=['age', 'bmi', 'sex'])
      .add_p()
      .theme('clinical')
)

SofraTable(rows=6, cols=4, theme='clinical')

In [25]:
(
    ps.tbl_one(df, by='arm', missing='never',
               variables=['age', 'bmi', 'sex'])
      .add_p()
      .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

## 8. `tbl_summary` — descriptive summaries without stratification

Same engine as `tbl_one`, but without a `by=` column. Useful for a
one-shot overall summary of a cohort or sub-cohort.

---


In [26]:
(
    ps.tbl_summary(
        df,
        variables=['age', 'bmi', 'sex', 'smoker', 'race'],
        labels={'age': 'Age (years)', 'bmi': 'BMI (kg/m\u00b2)'},
    )
    .set_caption('Cohort description (N=300)')
    .theme('compact')
)

Characteristic,OverallN = 300
Age (years),61.79 (10.42)
BMI (kg/m²),28.08 (4.97)
Missing,8 (2.7%)
sex = Male,131 (43.7%)
smoker = 1,97 (32.3%)
race,
Asian,55 (18.3%)
Black,61 (20.3%)
Other,16 (5.3%)
White,168 (56.0%)


## 9. Regression results — logistic

> Any fitted `statsmodels`, `lifelines`, or `sklearn` model can be
> passed to `tbl_regression()`.

The estimate column auto-labels to **OR** when `exponentiate=True` is
combined with a logit-family model. The 95% CI is computed on the
log-odds scale and exponentiated for display.


In [27]:
df['sex_M'] = (df['sex'] == 'Male').astype(int)
fit_df = df.dropna(subset=['age', 'bmi', 'sex_M', 'event'])
X = sm.add_constant(fit_df[['age', 'bmi', 'sex_M']])
logit_fit = sm.Logit(fit_df['event'], X).fit(disp=False)

(
    ps.tbl_regression(
        logit_fit,
        exponentiate=True,
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .set_caption('Logistic regression — adjusted odds ratios')
    .theme('jama')
)

SofraTable(rows=3, cols=4, theme='jama')

## 10. Regression results — OLS (linear)

For an OLS model the estimate column auto-labels to **β** and CIs are
reported on the natural scale.


In [28]:
df_complete = df.dropna(subset=['bmi'])
X = sm.add_constant(df_complete[['age', 'sex_M']])
ols_fit = sm.OLS(df_complete['bmi'], X).fit()

(
    ps.tbl_regression(
        ols_fit,
        labels={'age': 'Age (per year)', 'sex_M': 'Male sex'},
    )
    .set_caption('Linear regression of BMI')
    .theme('nejm')
)

SofraTable(rows=2, cols=4, theme='nejm')

## 11. Highlighting significant effects — `bold_p`

`bold_p()` bolds any body row whose p-value crosses the threshold
(default 0.05). Works on any `SofraTable` that has a p-value column.


In [29]:
(
    ps.tbl_regression(
        logit_fit,
        exponentiate=True,
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m\u00b2)',
                'sex_M': 'Male sex'},
    )
    .bold_p(threshold=0.05)
    .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

## 12. Composing tables side-by-side — `tbl_merge`

Two Table 1s — one per sex stratum — fused into a single deliverable
with spanning headers. Useful for clinical journal tables that report
per-subgroup *and* overall.


In [30]:
vars_ = ['age', 'bmi', 'smoker']

t_f = (
    ps.tbl_one(df[df.sex == 'Female'], by='arm',
               variables=vars_, missing='never')
      .add_p()
)
t_m = (
    ps.tbl_one(df[df.sex == 'Male'], by='arm',
               variables=vars_, missing='never')
      .add_p()
)

(
    ps.tbl_merge([t_f, t_m], tab_spanners=['Female', 'Male'])
      .set_caption('Baseline characteristics, by sex stratum')
      .theme('clinical')
)

SofraTable(rows=3, cols=7, theme='clinical')

## 13. Stacking tables vertically — `tbl_stack`

Concatenate two cohort tables vertically with group-header rows.
Useful when you want a single deliverable that covers, for example,
both a discovery and a validation cohort.


In [31]:
vars_ = ['age', 'bmi', 'smoker']

t1 = ps.tbl_one(df.iloc[:150], by='arm', variables=vars_,
                missing='never').add_p()
t2 = ps.tbl_one(df.iloc[150:], by='arm', variables=vars_,
                missing='never').add_p()

(
    ps.tbl_stack([t1, t2], group_labels=['Cohort A (n=150)',
                                          'Cohort B (n=150)'])
      .set_caption('Pooled cohort summary')
      .theme('clinical')
)

SofraTable(rows=8, cols=4, theme='clinical')

## 14. Per-variable test overrides

PySofra's auto-selection is right most of the time, but for sparse
categorical contingencies or strongly skewed continuous variables
you may want to pin a specific test. Pass
`tests={'var': 'test_name'}` — the footnote will list both the
overrides and what was used for the others.

**Available tests:**
`welch`, `student`, `wilcoxon`, `anova`, `kruskal` (continuous);
`fisher`, `chisq` (categorical).


In [32]:
(
    ps.tbl_one(
        df, by='arm',
        variables=['age', 'bmi', 'hba1c', 'sex', 'race'],
        tests={
            'hba1c': 'wilcoxon',   # known skew → rank-based
            'race':  'fisher',     # small per-cell counts → exact
        },
    )
    .add_p()
    .theme('clinical')
)

Characteristic,PlaceboN = 145,TreatmentN = 155,p-value
age,61.18 (10.68),62.36 (10.17),0.330
bmi,28.38 (4.94),27.80 (5.00),0.321
Missing,2 (1.4%),6 (3.9%),
hba1c,6.62 (1.11),6.57 (1.18),0.616
sex = Male,61 (42.1%),70 (45.2%),0.642
race,,,0.762
Asian,27 (18.6%),28 (18.1%),
Black,32 (22.1%),29 (18.7%),
Other,6 (4.1%),10 (6.5%),
White,80 (55.2%),88 (56.8%),


## 15. Multiplicity adjustment — `.add_q()`

When you're comparing many variables, raw p-values mislead.
`.add_q(method='fdr_bh')` (default) appends a Benjamini–Hochberg
adjusted q-value column.

**Supported methods:** `fdr_bh`, `fdr_by`, `bonferroni`, `holm`,
`hommel`, `sidak`.


In [33]:
(
    ps.tbl_one(df, by='arm',
               variables=['age', 'bmi', 'hba1c', 'sex', 'smoker', 'race'])
      .add_p()
      .add_q(method='fdr_bh')
      .theme('clinical')
)

Characteristic,PlaceboN = 145,TreatmentN = 155,p-value,q-value
age,61.18 (10.68),62.36 (10.17),0.330,0.659
bmi,28.38 (4.94),27.80 (5.00),0.321,0.659
Missing,2 (1.4%),6 (3.9%),,
hba1c,6.62 (1.11),6.57 (1.18),0.735,0.749
sex = Male,61 (42.1%),70 (45.2%),0.642,0.749
smoker = 1,52 (35.9%),45 (29.0%),0.219,0.659
race,,,0.749,0.749
Asian,27 (18.6%),28 (18.1%),,
Black,32 (22.1%),29 (18.7%),,
Other,6 (4.1%),10 (6.5%),,


## 16. Side-by-side multi-model regression

Pass a list of fitted models to get a side-by-side comparison. Spanning
headers auto-label each model; the coefficient set is the *union* of
all models, with em-dashes where a coefficient isn't in a given model.


In [34]:
X_uni = sm.add_constant(fit_df[['age']])
fit_uni = sm.Logit(fit_df['event'], X_uni).fit(disp=False)

(
    ps.tbl_regression(
        [fit_uni, logit_fit],
        exponentiate=True,
        model_labels=['Unadjusted', 'Adjusted'],
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .set_caption('Unadjusted vs. adjusted odds ratios')
    .theme('jama')
)

SofraTable(rows=3, cols=7, theme='jama')

## 17. Survival regression — Cox (lifelines)

`tbl_regression()` natively accepts `lifelines.CoxPHFitter` and the
AFT family. Hazard ratios with CIs are emitted automatically when
`exponentiate=True`.


In [35]:
from lifelines import CoxPHFitter

surv = df.dropna(subset=['age', 'bmi']).copy()
surv['time']  = np.random.default_rng(2026).exponential(scale=10, size=len(surv))
surv['died']  = surv['event']

cph = CoxPHFitter()
cph.fit(surv[['age', 'bmi', 'sex_M', 'time', 'died']],
        duration_col='time', event_col='died')

(
    ps.tbl_regression(
        cph,
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .set_caption('Cox proportional hazards model')
    .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

## 18. Polars input

PySofra accepts `pandas.DataFrame`, `polars.DataFrame`, and
`polars.LazyFrame`. The input adapter normalises to pandas internally;
the rendered table is byte-identical to the pandas path.


In [36]:
import polars as pl

pl_df = pl.from_pandas(df.drop(columns='sex_M'))

(
    ps.tbl_one(pl_df, by='arm',
               variables=['age', 'bmi', 'sex'])
      .add_p()
      .theme('clinical')
)

SofraTable(rows=4, cols=4, theme='clinical')

## 19. Kaplan–Meier summary — `tbl_survival`

A publication-ready KM table in a single call:

* N total / events / censored per arm
* Median survival with confidence interval
* Survival probability at user-chosen time points (with N at risk)
* Multivariate log-rank p-value across arms


In [37]:
surv_df = df.copy()
surv_df['time'] = np.random.default_rng(2026).exponential(scale=24, size=len(surv_df))
surv_df['died'] = surv_df['event']

(
    ps.tbl_survival(
        surv_df,
        time='time',
        event='died',
        by='arm',
        times=[6, 12, 24],
        times_label='mo',
        labels={'Placebo': 'Placebo arm', 'Treatment': 'Treatment arm'},
    )
    .set_caption('Kaplan–Meier survival summary, by arm')
    .theme('clinical')
)

SofraTable(rows=7, cols=4, theme='clinical')

## 20. Survey-weighted Table 1

Pass `weights='colname'` to get frequency-weighted means, variances,
and proportions. Useful for IPW-adjusted descriptive summaries or any
context where the sample requires re-weighting before reporting.


In [38]:
weighted_df = df.copy()
weighted_df['ipw'] = np.random.default_rng(11).uniform(0.5, 2.5, size=len(weighted_df))

(
    ps.tbl_one(
        weighted_df,
        by='arm',
        variables=['age', 'bmi', 'sex', 'smoker'],
        weights='ipw',
        labels={'age': 'Age (years)', 'bmi': 'BMI (kg/m²)',
                'sex': 'Sex', 'smoker': 'Current smoker'},
    )
    .add_p()
    .set_caption('IPW-weighted baseline characteristics')
    .theme('clinical')
)

SofraTable(rows=5, cols=4, theme='clinical')

## 21. Complex survey designs — `SurveyDesign`

For stratified and/or clustered designs, wrap the design columns in a
`SurveyDesign` and pass it via `design=`. The variance estimator
switches to Taylor linearisation (mean + design-based SE), and
categorical tests automatically use the Rao–Scott corrected χ².


In [39]:
surv_design_df = df.copy()
surv_design_df['region'] = np.random.default_rng(101).choice(
    ['Northeast', 'South', 'West', 'Midwest'], size=len(surv_design_df)
)
surv_design_df['psu'] = np.random.default_rng(102).choice(
    range(40), size=len(surv_design_df)
)
surv_design_df['pweight'] = np.random.default_rng(103).uniform(
    0.5, 3.0, size=len(surv_design_df)
)

design = ps.SurveyDesign(
    weights='pweight',
    strata='region',
    cluster='psu',
)

(
    ps.tbl_one(
        surv_design_df, by='arm',
        variables=['age', 'bmi', 'sex', 'smoker'],
        design=design,
        labels={'age': 'Age (years)', 'bmi': 'BMI (kg/m²)',
                'sex': 'Sex', 'smoker': 'Current smoker'},
    )
    .add_p()
    .set_caption('Stratified + clustered survey design')
    .theme('clinical')
)

/Users/jasonturner/Projects/pysofra/src/pysofra/summary/tbl_one.py:411: UserWarning: Rao–Scott chi-square for 'sex': pysofra uses the first-order Kish-DEFF approximation which does not account for stratification or clustering in the provided SurveyDesign. The reported p-value may disagree with R ``survey::svychisq`` by 10–15% or more. For design-grade chi-square inference on complex surveys, call ``survey::svychisq`` in R.
  row_blocks, test_used = _categorical_rows(
/Users/jasonturner/Projects/pysofra/src/pysofra/summary/tbl_one.py:411: UserWarning: Rao–Scott chi-square for 'smoker': pysofra uses the first-order Kish-DEFF approximation which does not account for stratification or clustering in the provided SurveyDesign. The reported p-value may disagree with R ``survey::svychisq`` by 10–15% or more. For design-grade chi-square inference on complex surveys, call ``survey::svychisq`` in R.
  row_blocks, test_used = _categorical_rows(


SofraTable(rows=5, cols=4, theme='clinical')

### 21.1 Design-adjusted t-test + cluster-robust regression

When a `SurveyDesign` with strata or cluster is supplied, `add_p()`
auto-routes continuous tests to a design-adjusted *t*-test (an
`svyttest` analogue). For regression, pass `design=` *and* `data=` to
`tbl_regression()` and the model is re-fit with cluster-robust (or
HC1) standard errors — matching R's `survey::svyglm` to first order.

---


In [40]:
# Build a logit model and re-fit with cluster-robust SEs using design=
import statsmodels.formula.api as smf

reg_df = surv_design_df.copy()
reg_df['sex_M'] = (reg_df['sex'] == 'Male').astype(int)
reg_df = reg_df.dropna(subset=['age', 'bmi', 'sex_M', 'event'])

fit = smf.logit('event ~ age + bmi + sex_M', data=reg_df).fit(disp=False)

design = ps.SurveyDesign(weights='pweight', strata='region', cluster='psu')

(
    ps.tbl_regression(
        fit, design=design, data=reg_df,
        exponentiate=True,
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .set_caption('Logistic regression with cluster-robust SE (Taylor-linearised sandwich)')
    .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

## 22. Univariable regression — `tbl_uvregression`

One regression per predictor, stacked into a single table. Pass
`adjust_for=` to control for a fixed set of confounders in every row
(producing a partially-adjusted univariable analysis common in
manuscript Table 2s).


In [41]:
(
    ps.tbl_uvregression(
        fit_df,
        outcome='event',
        predictors=['age', 'bmi', 'sex_M'],
        method='Logit',
        exponentiate=True,
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .set_caption('Univariable logistic regression — one OR per predictor')
    .theme('clinical')
)

SofraTable(rows=3, cols=5, theme='clinical')

## 23. Differences, CIs, joint p-values

Three modifiers that enrich a Table 1 in place:

* **`.add_difference()`** — between-group difference (with CI) column,
  for 2-group Table 1s.
* **`.add_ci()`** — appends a CI to every summary cell.
* **`.add_global_p()`** — reserves a column for joint Type-III tests
  on regression-style tables.


In [42]:
(
    ps.tbl_one(df, by='arm', variables=['age', 'bmi', 'sex'],
               missing='never',
               labels={'age': 'Age (years)', 'bmi': 'BMI (kg/m²)', 'sex': 'Sex'})
      .add_p()
      .add_difference()
      .add_ci()
      .set_caption('Baseline characteristics with differences and 95% CIs')
      .theme('clinical')
)

SofraTable(rows=3, cols=5, theme='clinical')

## 24. Custom formatters & in-prose extraction

For polishing the last 5% of a manuscript-ready table:

* **`with_pvalue_fmt(fn)`** / **`with_estimate_fmt(fn)`** — plug in
  your own formatters (e.g. four-decimal p-values).
* **`inline_text(row, column)`** — pull a single rendered cell for
  in-prose use (e.g. *"…mean (SD) 56.2 (12.4) in the Placebo arm…"*).
* **`modify_spanning_header(label, start, end)`** — attach a header
  span by hand.


In [43]:
demo = (
    ps.tbl_one(df, by='arm', variables=['age', 'bmi'], missing='never')
      .add_p()
      .add_smd()
      .with_pvalue_fmt(lambda p: f"p = {p:.4f}")
      .modify_spanning_header('Treatment groups', start=1, end=2)
      .theme('clinical')
)
# Display
display(demo)
# Inline extraction
print('In-prose: the adjusted p-value for age was', demo.inline_text(row='age', column='p-value'))

SofraTable(rows=2, cols=5, theme='clinical')

In-prose: the adjusted p-value for age was p = 0.3296


## 25. Visual previews — table-as-image + KM plot

* **`.to_image(path)`** renders the *table itself* to PNG. Useful for
  Slack previews, README screenshots, or pre-flight checks.
* **`with_km_plot(risk_times=[...])`** attaches a Kaplan–Meier curve
  *with an N-at-risk grid below it* — the same plot is embedded
  consistently across HTML, DOCX, PPTX, and LaTeX exports.


In [44]:
from pathlib import Path

preview_tbl = (
    ps.tbl_one(df, by='arm', variables=['age', 'bmi', 'sex'], missing='never')
      .add_p().add_smd().theme('clinical')
)
img_path = Path('tutorial_preview.png')
preview_tbl.to_image(img_path)
print(f'Wrote {img_path}  ({img_path.stat().st_size:,} bytes)')

(
    ps.tbl_survival(
        surv_df, time='time', event='died', by='arm',
        times=[6, 12, 24], times_label='mo',
    )
    .with_km_plot(risk_times=[0, 6, 12, 18, 24, 30])
    .set_caption('KM with N-at-risk table below')
    .theme('clinical')
)

Wrote tutorial_preview.png  (120,799 bytes)


SofraTable(rows=7, cols=4, theme='clinical')

## 26. Rich-cell composition — `compose()` + `CellPart`

For a single cell, you can push multiple typographically distinct
runs into one string — bold + italic + superscript + colour + link —
all in one cell. Renderers honour what they support and gracefully
fall back to concatenated plain text on backends that don't.


In [45]:
rich = (
    ps.tbl_one(df, by='arm', variables=['age', 'bmi'], missing='never')
      .add_p().add_smd()
      .compose(0, 0, [
          ps.CellPart('Age', bold=True),
          ps.CellPart(' (years)', italic=True),
          ps.CellPart(' ★', color='#c1272d', superscript=True),
      ])
      .compose(0, 1, [
          ps.CellPart('μ', italic=True),
          ps.CellPart(' = '),
          ps.CellPart('62.5', bold=True),
      ])
      .theme('clinical')
)
rich

SofraTable(rows=2, cols=5, theme='clinical')

## 27. Calibration — `post_stratify` / `rake`

For survey samples that need to match known population marginals,
calibrate weights *before* passing them to `SurveyDesign`:

* **`post_stratify`** — exact solver for complete cross-classification
  (full joint cell counts known).
* **`rake`** — iterative proportional fitting (IPF) for the more
  common case where only marginal totals are available.

---


In [46]:
cal_df = surv_design_df.copy()
cal_df['base_w'] = 1.0

# Suppose the survey-population totals are:
#   Northeast: 400,  South: 350,  West: 150,  Midwest: 100  (1000 total)
#   Female: 520,     Male: 480
region_targets = {'Northeast': 400.0, 'South': 350.0, 'West': 150.0, 'Midwest': 100.0}
sex_targets    = {'Female': 520.0, 'Male': 480.0}

calibrated = ps.rake(
    cal_df, 'base_w',
    margins={'region': region_targets, 'sex': sex_targets},
)
cal_df['cal_w'] = calibrated

print('Region marginals after raking:')
for region, target in region_targets.items():
    got = calibrated[cal_df['region'] == region].sum()
    print(f'  {region:<10} target={target:>6.0f}  got={got:>8.3f}')

print()
print(f"Design effect (calibrated): {ps.design_effect(calibrated):.4f}")

# Plug into a SurveyDesign and render
design = ps.SurveyDesign(weights='cal_w', strata='region')
(
    ps.tbl_one(
        cal_df, by='arm',
        variables=['age', 'bmi', 'sex'],
        design=design,
        labels={'age': 'Age (years)', 'bmi': 'BMI (kg/m²)', 'sex': 'Sex'},
    )
    .add_p()
    .set_caption('Raked + stratified Table 1')
    .theme('clinical')
)

Region marginals after raking:
  Northeast  target=   400  got= 400.000
  South      target=   350  got= 350.000
  West       target=   150  got= 150.000
  Midwest    target=   100  got= 100.000

Design effect (calibrated): 1.2074


/Users/jasonturner/Projects/pysofra/src/pysofra/summary/tbl_one.py:411: UserWarning: Rao–Scott chi-square for 'sex': pysofra uses the first-order Kish-DEFF approximation which does not account for stratification or clustering in the provided SurveyDesign. The reported p-value may disagree with R ``survey::svychisq`` by 10–15% or more. For design-grade chi-square inference on complex surveys, call ``survey::svychisq`` in R.
  row_blocks, test_used = _categorical_rows(


SofraTable(rows=4, cols=4, theme='clinical')

## 28. Cross-tabulation — `tbl_cross`

Explicit two-way contingency tables with selectable cell content:
`n`, `row_pct`, `col_pct`, `total_pct`, or any `n (xx%)` combination.
A spanning header names the column variable, and row/column margins
are included by default.


In [47]:
(
    ps.tbl_cross(
        df, row='sex', column='arm', cell='n_col_pct',
        labels={'Female': 'Women', 'Male': 'Men'},
    )
    .set_caption('Sex by treatment arm')
    .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

## 29. Effect sizes & one-line decorators

Effect-size functions and one-call table decorators that round out
the descriptive workflow:

* **Continuous:** `cohen_d`, `hedges_g` (small-sample corrected),
  `eta_squared`, `omega_squared`.
* **Categorical:** `cramers_v`, `phi_coefficient`.
* **Auto-dispatch:** `auto_effect_size()` mirrors the auto-test logic.
* **Decorators:** `.add_n()` (per-row N column), `.add_stat_label()`
  (a "Mean (SD)" / "n (%)" stat-label column),
  `.add_significance_stars()` (`*** ** *` markers),
  `color_scale_if()` (HTML heatmap on a numeric column).


In [48]:
a = df.loc[df.arm == 'Placebo', 'age']
b = df.loc[df.arm == 'Treatment', 'age']
print(f"Cohen's d (age, by arm): {ps.cohen_d(a, b):.4f}")
print(f"Hedges' g (age, by arm): {ps.hedges_g(a, b):.4f}")
print(f"η² (age, by race):       {ps.eta_squared(df['age'], df['race']):.4f}")
print(f"Cramér's V (sex, arm):   {ps.cramers_v(df['sex'], df['arm']):.4f}")

(
    ps.tbl_one(df, by='arm',
               variables=['age', 'bmi', 'sex', 'smoker'],
               missing='never',
               labels={'age': 'Age (years)', 'bmi': 'BMI (kg/m²)',
                       'sex': 'Sex', 'smoker': 'Current smoker'})
      .add_p()
      .add_smd()
      .add_n()
      .add_stat_label()
      .add_significance_stars()
      .color_scale_if(column=5, palette=('#f7fbff', '#9ecae1', '#08519c'))
      .set_caption('Baseline characteristics with N, stat label, stars, and SMD heatmap')
      .theme('clinical')
)

Cohen's d (age, by arm): -0.1130
Hedges' g (age, by arm): -0.1127
η² (age, by race):       0.0161
Cramér's V (sex, arm):   0.0312


SofraTable(rows=4, cols=8, theme='clinical')

## 30. Multiple-imputation pooling (Rubin's rules)

When you fit one model per imputed dataset (m ≥ 2), `ps.pool(fits)`
combines them via **Rubin's rules**: within-imputation +
between-imputation variance, Barnard–Rubin degrees of freedom. The
pooled `ModelSummary` plugs straight into `tbl_regression()` so the
output looks identical to a single-model table — with valid pooled
p-values.

---


In [49]:
# Simulate 5 imputations of `bmi` (it has some NaNs)
mi_base = df.dropna(subset=['age', 'sex_M']).copy()
mi_fits = []
rng_mi = np.random.default_rng(2026)
for _ in range(5):
    df_i = mi_base.copy()
    # Impute bmi from age (simple regression imputation + noise)
    mean_age = df_i['age'].mean()
    df_i['bmi_imp'] = np.where(
        df_i['bmi'].isna(),
        27 + 0.05 * (df_i['age'] - mean_age) + rng_mi.normal(0, 4, len(df_i)),
        df_i['bmi'],
    )
    X = sm.add_constant(df_i[['age', 'bmi_imp', 'sex_M']])
    mi_fits.append(sm.Logit(df_i['event'], X).fit(disp=False))

pooled = ps.pool(mi_fits)
(
    ps.tbl_regression(
        pooled,
        exponentiate=True,
        labels={'age': 'Age (per year)',
                'bmi_imp': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .set_caption('Pooled logistic regression across 5 multiple imputations')
    .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

## 31. Themes

Six built-in themes — *same data, six looks*. Use `.theme(name)` to
switch. `clinical` is the default for trial-style tables; `jama` and
`nejm` mimic the typographic conventions of those journals.


In [50]:
from IPython.display import HTML, display

base = ps.tbl_one(
    df, by='arm',
    variables=['age', 'bmi', 'sex', 'smoker'],
    missing='never',
).add_p().add_smd()

for theme in ['clinical', 'compact', 'jama', 'nejm', 'minimal']:
    display(HTML(f'<h4 style="margin-top:1em">Theme: <code>{theme}</code></h4>'))
    display(base.theme(theme))

SofraTable(rows=4, cols=5, theme='clinical')

SofraTable(rows=4, cols=5, theme='compact')

SofraTable(rows=4, cols=5, theme='jama')

SofraTable(rows=4, cols=5, theme='nejm')

SofraTable(rows=4, cols=5, theme='minimal')

## 32. Captions, footnotes, and custom themes

Captions and footnotes are first-class properties of every
`SofraTable`. Custom themes can be registered with `register_theme()`
and used by name everywhere — including in DOCX and LaTeX exports.


In [51]:
from pysofra.themes.registry import Theme, register_theme

house_style = Theme(
    name='house',
    css={
        'table': {'font-family': '"Inter", "Helvetica Neue", sans-serif',
                  'font-size': '13px', 'border-collapse': 'collapse'},
        'caption': {'font-weight': '700', 'font-size': '16px',
                    'caption-side': 'top', 'text-align': 'left',
                    'padding': '0.4em 0', 'color': '#0b3d91'},
        'th': {'background': '#0b3d91', 'color': 'white',
               'padding': '0.55em 0.85em', 'text-align': 'center'},
        'td': {'padding': '0.4em 0.85em',
               'border-bottom': '1px solid rgba(127,127,127,0.25)'},
        'tfoot td': {'font-size': '11px', 'color': '#555',
                     'border-bottom': 'none', 'padding-top': '0.5em'},
    },
    docx={'font_name': 'Inter', 'font_size': 10, 'header_bottom_border': True},
)
register_theme(house_style)

(
    ps.tbl_one(df, by='arm',
               variables=['age', 'bmi', 'sex', 'smoker'],
               missing='never')
      .add_p()
      .add_smd()
      .set_caption('Table 1. Baseline characteristics, by treatment arm')
      .add_footnote('Data: synthetic trial, n=300 randomised 1:1.')
      .theme('house')
)

SofraTable(rows=4, cols=5, theme='house')

## 33. Conditional row formatting

Beyond `bold_p`, PySofra exposes general row-level conditional styling:

* **`bold_if(predicate)`** — bold rows where `predicate(row)` is True.
* **`highlight_if(predicate, color='#fff3cd')`** — set a row background.
* **`style_if(predicate, bold=, italic=, color=)`** — combine all three.

`predicate` is a callable that receives the row and returns a bool.


In [52]:
(
    ps.tbl_one(df, by='arm',
               variables=['age', 'bmi', 'sex', 'smoker', 'race', 'hba1c'],
               missing='never')
      .add_p()
      .style_if(lambda r: r.cells[0].text.startswith('age'),
                bold=True, color='#fff3cd')
      .highlight_if(lambda r: r.cells[0].text.startswith('hba1c'),
                    color='#e6f4ea')
      .theme('clinical')
)

Characteristic,PlaceboN = 145,TreatmentN = 155,p-value
age,61.18 (10.68),62.36 (10.17),0.330
bmi,28.38 (4.94),27.80 (5.00),0.321
sex = Male,61 (42.1%),70 (45.2%),0.642
smoker = 1,52 (35.9%),45 (29.0%),0.219
race,,,0.749
Asian,27 (18.6%),28 (18.1%),
Black,32 (22.1%),29 (18.7%),
Other,6 (4.1%),10 (6.5%),
White,80 (55.2%),88 (56.8%),
hba1c,6.62 (1.11),6.57 (1.18),0.735


## 34. Sticky headers for long tables

For long tables inside a notebook, pair `sticky_header=True` with a
`max_height` so the column headers stay in view while the body scrolls.


In [53]:
from IPython.display import HTML

# Repeat a large variable list to demonstrate scrolling.
big_vars = ['age', 'bmi', 'hba1c', 'sex', 'smoker', 'race']
big_tbl = (
    ps.tbl_one(df, by='arm', variables=big_vars)
      .add_p()
      .add_smd()
      .theme('clinical')
)
HTML(big_tbl.to_html(sticky_header=True, max_height='240px'))

Characteristic,PlaceboN = 145,TreatmentN = 155,p-value,SMD
age,61.18 (10.68),62.36 (10.17),0.330,0.113
bmi,28.38 (4.94),27.80 (5.00),0.321,0.116
Missing,2 (1.4%),6 (3.9%),,
hba1c,6.62 (1.11),6.57 (1.18),0.735,0.039
sex = Male,61 (42.1%),70 (45.2%),0.642,0.062
smoker = 1,52 (35.9%),45 (29.0%),0.219,0.146
race,,,0.749,0.128
Asian,27 (18.6%),28 (18.1%),,
Black,32 (22.1%),29 (18.7%),,
Other,6 (4.1%),10 (6.5%),,


## 35. Inline plots — forest + Kaplan–Meier

`tbl_regression(...).with_forest_plot()` attaches an inline SVG forest
plot of the displayed estimates and CIs.
`tbl_survival(...).with_km_plot()` does the same with Kaplan–Meier
curves and an N-at-risk grid. Both plots are embedded consistently
across HTML, DOCX, PPTX, and LaTeX outputs.


In [54]:
(
    ps.tbl_regression(
        logit_fit,
        exponentiate=True,
        labels={'age': 'Age (per year)',
                'bmi': 'BMI (per kg/m²)',
                'sex_M': 'Male sex'},
    )
    .with_forest_plot()
    .set_caption('Adjusted odds ratios with forest plot')
    .theme('clinical')
)

SofraTable(rows=3, cols=4, theme='clinical')

In [55]:
(
    ps.tbl_survival(
        surv_df,
        time='time',
        event='died',
        by='arm',
        times=[6, 12, 24],
        times_label='mo',
    )
    .with_km_plot()
    .set_caption('Kaplan–Meier survival with curves')
    .theme('clinical')
)

SofraTable(rows=7, cols=4, theme='clinical')

## 36. Exports

Every `SofraTable` renders to **six** formats from a single object.
The next blocks show each one in turn.

### 36.1 Markdown — `.to_markdown()`


In [56]:
tbl = (
    ps.tbl_one(df, by='arm',
               variables=['age', 'bmi', 'sex'],
               missing='never')
      .add_p()
      .add_smd()
      .set_caption('Baseline characteristics')
      .theme('clinical')
)

print(tbl.to_markdown())

**Baseline characteristics**

| Characteristic | Placebo · N = 145 | Treatment · N = 155 | p-value | SMD |
| :--- | :---: | :---: | :---: | :---: |
| age | 61.18 (10.68) | 62.36 (10.17) | 0.330 | 0.113 |
| bmi | 28.38 (4.94) | 27.80 (5.00) | 0.321 | 0.116 |
| sex = Male | 61 (42.1%) | 70 (45.2%) | 0.642 | 0.062 |

_Mean (SD) for continuous variables._
_n (%) for categorical variables._
_Tests: Fisher's exact; Welch's t-test._
_SMD = standardized mean difference (max pairwise)._



### 36.2 HTML fragment — `.to_html()`

The notebook representation (`_repr_html_`) wraps the same fragment
in a scrollable container so it scales gracefully inside cells.


In [57]:
print(tbl.to_html()[:600], '\n...')

<style>table.pysofra-e90d9512a5{border-collapse:collapse;font-family:"Helvetica Neue", Helvetica, Arial, "Segoe UI", "Liberation Sans", sans-serif;font-size:14px;line-height:1.45;color:inherit;background:transparent;margin:0.75em 0;}.pysofra-e90d9512a5 caption{caption-side:top;text-align:left;font-weight:700;padding:0.4em 0.2em;font-size:15px;color:inherit;}.pysofra-e90d9512a5 th{padding:0.55em 0.85em;text-align:center;border-top:2.5px solid currentColor;border-bottom:1.5px solid currentColor;font-weight:700;vertical-align:bottom;color:inherit;background:transparent;}.pysofra-e90d9512a5 td{pad 
...


### 36.3 Word — `.to_docx(path)`

Publication-quality `.docx` via `python-docx`. Theme styling,
spanning headers, and inline plots all carry through.


In [58]:
from pathlib import Path

out = Path('tutorial_table1.docx')
tbl.to_docx(out)
print(f'Wrote {out}  ({out.stat().st_size:,} bytes)')

Wrote tutorial_table1.docx  (37,194 bytes)


### 36.4 LaTeX — `.to_latex()`

Returns a self-contained `\begin{table} ... \end{table}` block using
the `booktabs` rule conventions. Requires `\usepackage{booktabs}` in
the consuming document preamble.


In [59]:
print(tbl.to_latex())

\begin{table}[ht]
\centering
\caption{Baseline characteristics}
\begin{tabular}{lcccc}
\toprule
\textbf{Characteristic} & \textbf{\shortstack{Placebo \\ N = 145}} & \textbf{\shortstack{Treatment \\ N = 155}} & \textbf{p-value} & \textbf{SMD} \\
\midrule
age & 61.18 (10.68) & 62.36 (10.17) & 0.330 & 0.113 \\
bmi & 28.38 (4.94) & 27.80 (5.00) & 0.321 & 0.116 \\
sex = Male & 61 (42.1\%) & 70 (45.2\%) & 0.642 & 0.062 \\
\bottomrule
\end{tabular}
\vspace{0.25em}
\par\noindent\small\textit{Mean (SD) for continuous variables.}
\par\noindent\small\textit{n (\%) for categorical variables.}
\par\noindent\small\textit{Tests: Fisher's exact; Welch's t-test.}
\par\noindent\small\textit{SMD = standardized mean difference (max pairwise).}
\end{table}



### 36.5 PowerPoint — `.to_pptx(path)`

Writes a single-slide deck. Requires the optional extra
(`pip install pysofra[pptx]`).


In [60]:
pptx_out = Path('tutorial_table1.pptx')
tbl.to_pptx(pptx_out, slide_title='Table 1 — Baseline characteristics')
print(f'Wrote {pptx_out}  ({pptx_out.stat().st_size:,} bytes)')

Wrote tutorial_table1.pptx  (29,052 bytes)


### 36.6 Excel — `.to_xlsx(path)`

Writes a styled single-sheet workbook via `xlsxwriter`. Numeric cells
are written as **actual numbers** (not strings), so downstream
analysts can run formulas on the export immediately.

---


In [61]:
xlsx_out = Path('tutorial_table1.xlsx')
tbl.to_xlsx(xlsx_out, sheet_name='Baseline')
print(f'Wrote {xlsx_out}  ({xlsx_out.stat().st_size:,} bytes)')

Wrote tutorial_table1.xlsx  (6,088 bytes)


## 37. Under the hood — the `SofraTable` object

Every `SofraTable` is a frozen dataclass. **Every modifier returns a
new instance**; the original is untouched. This means you can build a
base table once, then derive many variants without worrying about
state leaking between them.


In [62]:
base = ps.tbl_one(df, by='arm', variables=['age', 'sex'], missing='never')
with_p = base.add_p()

print('base headers:  ', [c.text.replace("\n", " | ") for c in base.headers[0].cells])
print('with_p headers:', [c.text.replace("\n", " | ") for c in with_p.headers[0].cells])
print('base is unchanged:', len(base.headers[0].cells) == 3)
print()
print('Theme:', with_p.theme_name)
print('Metadata:', with_p.metadata)

base headers:   ['Characteristic', 'Placebo | N = 145', 'Treatment | N = 155']
with_p headers: ['Characteristic', 'Placebo | N = 145', 'Treatment | N = 155', 'p-value']
base is unchanged: True

Theme: default
Metadata: {'builder': 'tbl_one', 'tests': ["Fisher's exact", "Welch's t-test"]}


### `to_dict()` — structural snapshot

Useful for testing, downstream consumers, and stable diffs in CI.


In [63]:
d = with_p.add_smd().to_dict()
print('Headers:', d['headers'])
print('First 3 rows:')
for row in d['rows'][:3]:
    print(' ', row)
print('Footnotes:', d['footnotes'])

Headers: [['Characteristic', 'Placebo\nN = 145', 'Treatment\nN = 155', 'p-value', 'SMD']]
First 3 rows:
  ['age', '61.18 (10.68)', '62.36 (10.17)', '0.330', '0.113']
  ['sex = Male', '61 (42.1%)', '70 (45.2%)', '0.642', '0.062']
Footnotes: ['Mean (SD) for continuous variables.', 'n (%) for categorical variables.', "Tests: Fisher's exact; Welch's t-test.", 'SMD = standardized mean difference (max pairwise).']


---

## What's in the box

PySofra (alpha `0.1.0a1`) ships:

* **Builders** — `tbl_one`, `tbl_summary`, `tbl_cross`, `tbl_regression`,
  `tbl_uvregression`, `tbl_survival`.
* **Composition** — `tbl_merge`, `tbl_stack`, rich cell parts.
* **Statistics** — auto-selected hypothesis tests, Rao–Scott χ²,
  design-adjusted *t*-tests, Cox / OLS / Logit regressions, Rubin's
  rules pooling, six effect sizes, Wilson / Welch CIs.
* **Survey** — `SurveyDesign` (strata + cluster + FPC + replicate
  weights), `post_stratify` / `rake` calibration.
* **Renderers** — HTML, Markdown, DOCX, LaTeX, PPTX, XLSX, PNG.
* **Themes** — six built-in (`clinical`, `compact`, `jama`, `nejm`,
  `minimal`) plus user-registered ones.
* **Quality** — 100% line coverage, mypy strict, Hypothesis property
  tests, every numeric output validated against scipy / lifelines /
  statsmodels.

Feedback, bug reports, and contributions are very welcome.

> **Citing PySofra:** see `CITATION.cff` in the repository root.
